# Chapter 12: Hybrid Search and Reciprocal Rank Fusion

Exact RRF, absent contributions and depth effects.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:
from document_identity import Collection
from evidence_units import build_units
from evidence_pipeline import identity, method_candidates, candidate_union, fuse

collection = Collection.load(ROOT / "examples" / "fixtures" / "part2-v1.json")
units = build_units(collection, "part2", "sections")
a = next(u for u in units if u["source_id"] == "toy:blog")
b = next(u for u in units if u["source_id"] == "toy:spec")
c = next(u for u in units if u["document_id"] == "b-focused")

bm25_hits = [{**u, "score": 3 - i} for i, u in enumerate([a, c])]
dense_hits = [{**u, "score": 3 - i} for i, u in enumerate([c, b])]
lists = {
    "bm25": method_candidates(bm25_hits, "bm25", "part2", "hand-test"),
    "dense": method_candidates(dense_hits, "dense", "part2", "hand-test"),
}

In [ ]:
rows = fuse(lists, 60)
shared = next(x for x in rows if identity(x["unit"]) == identity(c))
single = next(x for x in rows if identity(x["unit"]) == identity(a))
print("shared score:", shared["fusion_score"], "expected:", 1/62 + 1/61)
print("single score:", single["fusion_score"], "expected:", 1/61)
assert abs(shared["fusion_score"] - (1/62 + 1/61)) < 1e-12
assert abs(single["fusion_score"] - 1/61) < 1e-12
assert identity(rows[0]["unit"]) == identity(c)

In [ ]:
# Zero contribution for absent candidates; zero constant control on single-item lists.
control_lists = {
    "bm25": method_candidates([{**a, "score": 1.0}], "bm25", "part2", "hand-test"),
    "dense": method_candidates([{**b, "score": 1.0}], "dense", "part2", "hand-test"),
}
control0 = fuse(control_lists, 0)
print([x["fusion_score"] for x in control0])
assert all(x["fusion_score"] == 1.0 for x in control0)

In [ ]:
# Deeper lists can change ordering and demote a previously leading candidate.
shallow = fuse({
    "bm25": method_candidates([{**c, "score": 1.0}], "bm25", "part2", "hand-test"),
    "dense": method_candidates([{**c, "score": 1.0}], "dense", "part2", "hand-test"),
}, 60)
deep = fuse({
    "bm25": method_candidates([{**a, "score": 2.0}, {**c, "score": 1.0}], "bm25", "part2", "hand-test"),
    "dense": method_candidates([{**a, "score": 2.0}, {**c, "score": 1.0}], "dense", "part2", "hand-test"),
}, 60)
print("shallow first:", identity(shallow[0]["unit"]))
print("deep first:", identity(deep[0]["unit"]), "second:", identity(deep[1]["unit"]))
assert identity(shallow[0]["unit"]) == identity(c)
assert identity(deep[0]["unit"]) == identity(a)
assert identity(deep[1]["unit"]) == identity(c)

## Observed versus not established
Observed: exact RRF arithmetic, zero absent contributions, depth-dependent ordering. Not established: general hybrid superiority.